<a href="https://colab.research.google.com/github/krushna1133/NNDL-Practical/blob/main/NNDL_Practical10.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

# 1. Sample sales data
sales = pd.DataFrame({
    'CustomerID': [1,1,2,2,3,3,4,4,5,5,1,2,3,4,5],
    'ProductID':  ['A','B','A','C','B','C','A','D','C','D','C','B','A','B','A']
})

# 2. Encode customers and products
user_encoder = LabelEncoder()
item_encoder = LabelEncoder()

sales['user'] = user_encoder.fit_transform(sales['CustomerID'])
sales['item'] = item_encoder.fit_transform(sales['ProductID'])

num_users = sales['user'].nunique()
num_items = sales['item'].nunique()

# 3. Create positive and negative samples
positive = sales[['user', 'item']].drop_duplicates().copy()
positive['label'] = 1

purchased = set(zip(positive['user'], positive['item']))
negative = []

for u in range(num_users):
    for i in range(num_items):
        if (u, i) not in purchased:
            negative.append([u, i, 0])

negative = pd.DataFrame(
    negative, columns=['user', 'item', 'label']
)

data = pd.concat([positive, negative], ignore_index=True)
data = data.sample(frac=1, random_state=42)

# 4. Split the data
X = data[['user', 'item']].values
y = data['label'].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# 5. Build the deep learning model
user_input = tf.keras.Input(shape=(1,))
item_input = tf.keras.Input(shape=(1,))

user_embedding = tf.keras.layers.Embedding(
    num_users, 8
)(user_input)

item_embedding = tf.keras.layers.Embedding(
    num_items, 8
)(item_input)

user_vec = tf.keras.layers.Flatten()(user_embedding)
item_vec = tf.keras.layers.Flatten()(item_embedding)

x = tf.keras.layers.Concatenate()([user_vec, item_vec])
x = tf.keras.layers.Dense(16, activation='relu')(x)
x = tf.keras.layers.Dense(8, activation='relu')(x)
output = tf.keras.layers.Dense(1, activation='sigmoid')(x)

model = tf.keras.Model(
    inputs=[user_input, item_input],
    outputs=output
)

model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

# 6. Train the model
model.fit(
    [X_train[:, 0], X_train[:, 1]],
    y_train,
    epochs=30,
    batch_size=4,
    verbose=1
)

# 7. Evaluate the model
loss, accuracy = model.evaluate(
    [X_test[:, 0], X_test[:, 1]],
    y_test,
    verbose=0
)

print("Test Loss:", round(loss, 4))
print("Test Accuracy:", round(accuracy * 100, 2), "%")

# 8. Recommend products to a customer
def recommend_products(customer_id, top_n=3):
    if customer_id not in user_encoder.classes_:
        print("Customer not found!")
        return

    user = user_encoder.transform([customer_id])[0]

    bought = set(
        sales.loc[sales['user'] == user, 'item']
    )

    candidates = [
        i for i in range(num_items) if i not in bought
    ]

    if not candidates:
        print("No new products to recommend.")
        return

    users = np.full(len(candidates), user)

    scores = model.predict(
        [users, np.array(candidates)], verbose=0
    ).flatten()

    ranked = np.argsort(scores)[::-1][:top_n]

    print(f"\nRecommendations for Customer {customer_id}:")
    for idx in ranked:
        product_id = item_encoder.inverse_transform(
            [candidates[idx]]
        )[0]
        print(f"Product: {product_id}, Score: {scores[idx]:.4f}")

# 9. Display recommendations
recommend_products(1)

Epoch 1/30
4/4 ━━━━━━━━━━━━━━━━━━━━ 5s 18ms/step - accuracy: 0.3750 - loss: 0.6948
Epoch 2/30
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step - accuracy: 0.7500 - loss: 0.6911
Epoch 3/30
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step - accuracy: 0.7500 - loss: 0.6890
Epoch 4/30
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step - accuracy: 0.7500 - loss: 0.6877
Epoch 5/30
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 28ms/step - accuracy: 0.7500 - loss: 0.6855
Epoch 6/30
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step - accuracy: 0.7500 - loss: 0.6835 
Epoch 7/30
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 23ms/step - accuracy: 0.7500 - loss: 0.6816
Epoch 8/30
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step - accuracy: 0.7500 - loss: 0.6788
Epoch 9/30
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 34ms/step - accuracy: 0.7500 - loss: 0.6767 
Epoch 10/30
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step - accuracy: 0.7500 - loss: 0.6732
Epoch 11/30
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 40ms/step - accuracy: 0.7500 - loss: 0.6709
Epoch 12/30
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 23ms/step - accuracy: 0.7500 - loss: 0.6678